# Analise de Metadados

Autor:  Viviane da Rosa Sommer

Atualizado: 24/07/2025

Objetivo: O objetivo desse código é ler múltiplos arquivos CSV contendo metadados de vídeos de uma pasta, consolidar e analisar esses dados, gerar estatísticas descritivas e visualizações gráficas interativas sobre características dos vídeos (como duração, bitrate, resolução, codec, FPS, entre outros), e então compilar tudo isso em um relatório HTML.

## Importações necessárias
Lembre-se de reiniciar o kernel após as instalações!

In [1]:
!pip install pandas plotly jinja2
!pip install kaleido==0.2.1
!pip install plotly==5.19.0

In [2]:
import os
import pandas as pd
import plotly.express as px
import base64
from io import BytesIO
from typing import Dict, Any
from jinja2 import Environment

## Declaração de Constantes e Modelos

folder = Pasta com vários CSV's com metadados.
output = Nome do arquivo HTML que será gerado.

In [3]:
folder = 'metadados'
output = 'analise_metadados.html'

## Funções necessárias

In [38]:
def segundos_para_hms(segundos: float) -> str:
    if pd.isna(segundos):
        return "-"
    segundos = int(round(segundos))
    h = segundos // 3600
    m = (segundos % 3600) // 60
    s = segundos % 60
    return f"{h:02}:{m:02}:{s:02}"

def bytes_para_gb(x: float) -> float:
    if pd.isna(x):
        return 0.0
    return round(x / 1024 / 1024 / 1024, 2)

def format_float(x: float) -> str:
    if pd.isna(x):
        return "-"
    return f"{x:.2f}"

def padroniza_os(x):
    if pd.isna(x) or str(x).strip() == '':
        return '-'
    x = str(x).strip().upper().split('_')[0].split('-')[0]
    if not x.startswith('OS'):
        x = 'OS' + x
    if not x.startswith('OS00'):
        x = x.replace('OS', 'OS00', 1)
    return x

def load_csvs(folder: str) -> pd.DataFrame:
    arquivos = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith('.csv')]
    dfs = []
    for f in arquivos:
        df = pd.read_csv(f)
        df = df.dropna(how='all')
        dfs.append(df)
    df = pd.concat(dfs, ignore_index=True)
    df['OS'] = df['OS'].apply(padroniza_os)
    return df

def fig_to_base64(fig: Any) -> str:
    buf = BytesIO()
    fig.write_image(buf, format='png', width=900, height=500, scale=2)
    buf.seek(0)
    img_b64 = base64.b64encode(buf.read()).decode('utf-8')
    return f'data:image/png;base64,{img_b64}'

def make_plotly_figures(df: pd.DataFrame) -> dict:
    df = df.copy()
    df = df[~df.isin(['########']).any(axis=1)]
    df['duracao_segundos'] = pd.to_numeric(df['duracao_segundos'], errors='coerce')
    df['taxa_bit_total'] = pd.to_numeric(df['taxa_bit_total'], errors='coerce')
    df['largura'] = pd.to_numeric(df['largura'], errors='coerce')
    df['altura'] = pd.to_numeric(df['altura'], errors='coerce')
    df['taxa_frames'] = pd.to_numeric(df['taxa_frames'], errors='coerce')
    df['tamanho_arquivo_bytes'] = pd.to_numeric(df['tamanho_arquivo_bytes'], errors='coerce')
    figs = {}
    figs['duracao'] = fig_to_base64(
        px.histogram(
            df, x='duracao_segundos', nbins=30, title='Distribuição da Duração dos Vídeos',
            labels={'duracao_segundos': 'Duração (s)'},
            category_orders={'duracao_segundos': sorted(df['duracao_segundos'].dropna().unique())}
        ).update_yaxes(title_text='Quantidade')
    )
    figs['bitrate'] = fig_to_base64(
        px.histogram(
            df, x='taxa_bit_total', nbins=30, title='Distribuição do Bitrate Total',
            labels={'taxa_bit_total': 'Bitrate Total (bps)'},
            category_orders={'taxa_bit_total': sorted(df['taxa_bit_total'].dropna().unique())}
        ).update_yaxes(title_text='Quantidade')
    )
    figs['resolucao'] = fig_to_base64(
        px.scatter(
            df, x='largura', y='altura', color='codec',
            title='Resoluções dos Vídeos por Codec',
            labels={'largura': 'Largura (px)', 'altura': 'Altura (px)', 'codec': 'Codec'}
        )
    )
    figs['correlacao'] = fig_to_base64(
        px.scatter(
            df, x='duracao_segundos', y=df['tamanho_arquivo_bytes'].apply(bytes_para_gb),
            color='codec', title='Duração x Tamanho',
            labels={'duracao_segundos': 'Duração (s)', 'y': 'Tamanho (GB)', 'codec': 'Codec'}
        )
    )
    figs['bitrate_vs_pixels'] = fig_to_base64(
        px.scatter(
            df, x=df['largura'] * df['altura'], y='taxa_bit_total',
            color='codec', title='Total de Pixels x Bitrate Total',
            labels={'x': 'Total de Pixels', 'taxa_bit_total': 'Bitrate Total (bps)', 'codec': 'Codec'}
        )
    )
    figs['fps'] = fig_to_base64(
        px.histogram(
            df, x='taxa_frames', nbins=30, title='Distribuição de FPS',
            labels={'taxa_frames': 'FPS'},
            category_orders={'taxa_frames': sorted(df['taxa_frames'].dropna().unique())}
        ).update_yaxes(title_text='Quantidade')
    )
    return figs

def resumo(df: pd.DataFrame) -> dict:
    df = df.copy()
    df['duracao_segundos'] = pd.to_numeric(df['duracao_segundos'], errors='coerce')
    df['tamanho_arquivo_bytes'] = pd.to_numeric(df['tamanho_arquivo_bytes'], errors='coerce')
    df['taxa_bit_total'] = pd.to_numeric(df['taxa_bit_total'], errors='coerce')
    df['taxa_frames'] = pd.to_numeric(df['taxa_frames'], errors='coerce')
    stats = {}
    stats['total_arquivos'] = len(df)
    stats['os_unicas'] = df["OS"].nunique()
    stats['codecs'] = df['codec'].value_counts().rename_axis('Codec').reset_index(name='Quantidade')
    stats['profiles'] = df['perfil'].value_counts().rename_axis('Perfil').reset_index(name='Quantidade')
    stats['levels'] = df['nivel'].value_counts().rename_axis('Nível').reset_index(name='Quantidade')
    stats['color_spaces'] = df['espaco_cor'].value_counts().rename_axis('Espaço de Cor').reset_index(name='Quantidade')
    stats['resolucoes'] = df.groupby(['largura', 'altura']).size().sort_values(ascending=False).reset_index(name='Quantidade')
    stats['resolucoes'] = stats['resolucoes'].rename(columns={'largura': 'Largura (px)', 'altura': 'Altura (px)'})
    stats['duracao_media'] = segundos_para_hms(df['duracao_segundos'].mean())
    stats['tamanho_medio'] = format_float(bytes_para_gb(df['tamanho_arquivo_bytes'].mean()))
    stats['bitrate_medio'] = format_float(df['taxa_bit_total'].mean() / 1e6)
    stats['fps_medio'] = format_float(df['taxa_frames'].mean())
    stats['fps_min'] = format_float(df['taxa_frames'].min())
    stats['fps_max'] = format_float(df['taxa_frames'].max())
    stats['fps_por_codec'] = df.groupby('codec')['taxa_frames'].mean().reset_index()
    stats['fps_por_codec'] = stats['fps_por_codec'].rename(columns={'codec': 'Codec', 'taxa_frames': 'FPS Médio'})
    stats['fps_por_codec']['FPS Médio'] = stats['fps_por_codec']['FPS Médio'].apply(format_float)
    dur_mean = df['duracao_segundos'].mean()
    dur_std = df['duracao_segundos'].std()
    outliers = df[df['duracao_segundos'] > dur_mean + 3*dur_std][['OS', 'nome_arquivo', 'duracao_segundos']].copy()
    outliers = outliers.rename(columns={'OS': 'OS', 'nome_arquivo': 'Arquivo'})
    outliers['Duração'] = outliers['duracao_segundos'].apply(segundos_para_hms)
    stats['outliers'] = outliers[['OS', 'Arquivo', 'Duração']]
    df['pixels'] = df['largura'] * df['altura']
    stats['correlacao_pixels_bitrate'] = df[['pixels', 'taxa_bit_total']].corr().iloc[0, 1]
    top_longos = df[['OS', 'nome_arquivo', 'duracao_segundos']].sort_values('duracao_segundos', ascending=False).head(10).copy()
    top_longos = top_longos.rename(columns={'OS': 'OS', 'nome_arquivo': 'Arquivo'})
    top_longos['Duração'] = top_longos['duracao_segundos'].apply(segundos_para_hms)
    stats['top_longos'] = top_longos[['OS', 'Arquivo', 'Duração']]
    top_maiores = df[['OS', 'nome_arquivo', 'tamanho_arquivo_bytes']].sort_values('tamanho_arquivo_bytes', ascending=False).head(10).copy()
    top_maiores = top_maiores.rename(columns={'OS': 'OS', 'nome_arquivo': 'Arquivo', 'tamanho_arquivo_bytes': 'Tamanho (GB)'})
    top_maiores['Tamanho (GB)'] = top_maiores['Tamanho (GB)'].apply(bytes_para_gb).apply(format_float)
    stats['top_maiores'] = top_maiores[['OS', 'Arquivo', 'Tamanho (GB)']]
    resumo_codec = df.groupby('codec')[['duracao_segundos', 'tamanho_arquivo_bytes', 'taxa_bit_total', 'taxa_frames']].mean().reset_index()
    resumo_codec['Duração Média'] = resumo_codec['duracao_segundos'].apply(segundos_para_hms)
    resumo_codec['Tamanho Médio (GB)'] = resumo_codec['tamanho_arquivo_bytes'].apply(bytes_para_gb).apply(format_float)
    resumo_codec['Bitrate Médio (bps)'] = resumo_codec['taxa_bit_total'].apply(format_float)
    resumo_codec['FPS Médio'] = resumo_codec['taxa_frames'].apply(format_float)
    resumo_codec = resumo_codec.rename(columns={'codec': 'Codec'})
    stats['resumo_codec'] = resumo_codec[['Codec','Duração Média','Tamanho Médio (GB)','Bitrate Médio (bps)','FPS Médio']]
    resumo_resolucao = df.groupby(['largura','altura'])[['duracao_segundos','tamanho_arquivo_bytes','taxa_bit_total','taxa_frames']].mean().reset_index()
    resumo_resolucao['Duração Média'] = resumo_resolucao['duracao_segundos'].apply(segundos_para_hms)
    resumo_resolucao['Tamanho Médio (GB)'] = resumo_resolucao['tamanho_arquivo_bytes'].apply(bytes_para_gb).apply(format_float)
    resumo_resolucao['Bitrate Médio (bps)'] = resumo_resolucao['taxa_bit_total'].apply(format_float)
    resumo_resolucao['FPS Médio'] = resumo_resolucao['taxa_frames'].apply(format_float)
    resumo_resolucao = resumo_resolucao.rename(columns={'largura': 'Largura (px)', 'altura': 'Altura (px)'})
    stats['resumo_resolucao'] = resumo_resolucao[['Largura (px)','Altura (px)','Duração Média','Tamanho Médio (GB)','Bitrate Médio (bps)','FPS Médio']]
    por_os = df.groupby('OS').agg({
        'duracao_segundos': 'sum',
        'tamanho_arquivo_bytes': 'sum',
        'taxa_bit_total': 'mean',
        'taxa_frames': 'mean'
    }).reset_index()
    por_os['Duração Total'] = por_os['duracao_segundos'].apply(segundos_para_hms)
    por_os['Tamanho Total (GB)'] = por_os['tamanho_arquivo_bytes'].apply(bytes_para_gb).apply(format_float)
    por_os['Bitrate Médio (bps)'] = por_os['taxa_bit_total'].apply(format_float)
    por_os['FPS Médio'] = por_os['taxa_frames'].apply(format_float)
    por_os = por_os.rename(columns={'OS': 'OS'})
    stats['por_os'] = por_os[['OS','Duração Total','Tamanho Total (GB)','Bitrate Médio (bps)','FPS Médio']]
    for k in ['codecs', 'profiles', 'levels', 'color_spaces', 'resolucoes', 'fps_por_codec', 'outliers', 'top_longos', 'top_maiores', 'resumo_codec', 'resumo_resolucao', 'por_os']:
        if k in stats:
            stats[k] = stats[k].fillna('-')
    return stats

def render_html(stats: dict, figs: dict, output_path: str) -> None:
    template_str = """
    <!DOCTYPE html>
    <html lang="pt-br">
    <head>
    <meta charset="UTF-8">
    <title>Análise de Vídeos</title>
    <link href="https://fonts.googleapis.com/css?family=Montserrat:400,700&display=swap" rel="stylesheet">
    <style>
    body { font-family: 'Montserrat', Arial, sans-serif; background: #f6f8fa; color: #222; margin:0; text-align:center;}
    h1, h2 { color: #1a237e; text-align:center;}
    .container { max-width: 1100px; margin: 40px auto; background: #fff; border-radius: 10px; box-shadow: 0 4px 16px #0001; padding: 32px; text-align:center;}
    .section { margin-bottom: 40px; }
    table { border-collapse: collapse; width: 100%; margin: 16px 0; text-align:center;}
    th, td { border: 1px solid #ddd; padding: 8px; text-align:center;}
    th { background: #e3e7fd; text-align:center;}
    tr:nth-child(even) { background: #f1f5fb; }
    .highlight { background: #ffecb3; }
    .plot-img { margin-bottom: 24px; border-radius: 8px; box-shadow: 0 2px 8px #0002; background: #f7f9fb; padding: 12px; max-width: 100%; }
    </style>
    </head>
    <body>
    <div class="container">
    <h1>Análise dos Arquivos de Vídeo</h1>
    <div class="section">
    <h2>Resumo Geral</h2>
    <ul style="text-align: left;">
    <li><b>Total de arquivos:</b> {{ stats.total_arquivos }}</li>
    <li><b>OS únicas:</b> {{ stats.os_unicas }}</li>
    <li><b>Duração média:</b> {{ stats.duracao_media }}</li>
    <li><b>Tamanho médio:</b> {{ stats.tamanho_medio }} GB</li>
    <li><b>Bitrate médio:</b> {{ stats.bitrate_medio }} Mbps</li>
    <li><b>FPS médio:</b> {{ stats.fps_medio }}</li>
    </ul>
    </div>
    <div class="section">
    <h2>Distribuições e Gráficos</h2>
    <img class="plot-img" src="{{ figs.duracao }}">
    <img class="plot-img" src="{{ figs.bitrate }}">
    <img class="plot-img" src="{{ figs.resolucao }}">
    <img class="plot-img" src="{{ figs.correlacao }}">
    <img class="plot-img" src="{{ figs.bitrate_vs_pixels }}">
    <img class="plot-img" src="{{ figs.fps }}">
    </div>
    <div class="section">
    <h2>Distribuição de FPS por Codec</h2>
    {{ stats.fps_por_codec.to_html(classes='table', border=0, index=False) | safe }}
    </div>
    <div class="section">
    <h2>Codecs de Vídeo</h2>
    {{ stats.codecs.to_html(classes='table', border=0, index=False) | safe }}
    </div>
    <div class="section">
    <h2>Perfis de Codec</h2>
    {{ stats.profiles.to_html(classes='table', border=0, index=False) | safe }}
    </div>
    <div class="section">
    <h2>Níveis de Codec</h2>
    {{ stats.levels.to_html(classes='table', border=0, index=False) | safe }}
    </div>
    <div class="section">
    <h2>Espaços de Cor</h2>
    {{ stats.color_spaces.to_html(classes='table', border=0, index=False) | safe }}
    </div>
    <div class="section">
    <h2>Resoluções Mais Frequentes</h2>
    {{ stats.resolucoes.head(10).to_html(classes='table', border=0, index=False) | safe }}
    </div>
    <div class="section">
    <h2>Outliers em Duração</h2>
    {{ stats.outliers.to_html(classes='table', border=0, index=False) | safe }}
    </div>
    <div class="section">
    <h2>Top 10 Arquivos Mais Longos</h2>
    {{ stats.top_longos.to_html(classes='table', border=0, index=False) | safe }}
    </div>
    <div class="section">
    <h2>Top 10 Arquivos Maiores</h2>
    {{ stats.top_maiores.to_html(classes='table', border=0, index=False) | safe }}
    </div>
    <div class="section">
    <h2>Resumo por Codec</h2>
    {{ stats.resumo_codec.to_html(classes='table', border=0, index=False) | safe }}
    </div>
    <div class="section">
    <h2>Resumo por Resolução</h2>
    {{ stats.resumo_resolucao.head(10).to_html(classes='table', border=0, index=False) | safe }}
    </div>
    <div class="section">
    <h2>Resumo por OS</h2>
    {{ stats.por_os.to_html(classes='table', border=0, index=False) | safe }}
    </div>
    </div>
    </body>
    </html>
    """
    env = Environment()
    template = env.from_string(template_str)
    html = template.render(stats=stats, figs=figs)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(html)

## Gerar .html com análise dos metadados

In [39]:
df = load_csvs(folder)
figs = make_plotly_figures(df)
stats = resumo(df)
render_html(stats, figs, output)